# 02 — Integración de la red en los snapshots BBN

Objetivo: integrar la red desde \(t=50\ \mathrm{s}\) hasta \(t=1000\ \mathrm{s}\), usando una historia impuesta \(T(t)\), \(\rho(t)\).

Usamos interpolación logarítmica entre los tres snapshots del enunciado. Para un ejercicio básico esto es más limpio que mantener \(T,\rho\) constantes por tramos.

In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import bbn_network as bbn
importlib.reload(bbn)

snapshots = pd.read_csv("bbn_snapshots.csv")
snapshots

In [ ]:
# Historia T(t), rho(t): interpolación log-log.
t_nodes = snapshots["t_s"].to_numpy(dtype=float)
T9_nodes = snapshots["T9"].to_numpy(dtype=float)
rho_nodes = snapshots["rho_g_cm3"].to_numpy(dtype=float)

def interp_loglog(t, x_nodes, y_nodes):
    t = np.asarray(t, dtype=float)
    return np.exp(np.interp(np.log(t), np.log(x_nodes), np.log(y_nodes)))

def T9_of_t(t):
    return interp_loglog(t, t_nodes, T9_nodes)

def T_of_t(t):
    return 1.0e9 * T9_of_t(t)

def rho_of_t(t):
    return interp_loglog(t, t_nodes, rho_nodes)

# Comprobación: debe recuperar exactamente los snapshots.
check = snapshots.copy()
check["T9_interp"] = T9_of_t(t_nodes)
check["rho_interp"] = rho_of_t(t_nodes)
check

In [ ]:
# Inicialización de abundancias.
# En pynucastro se integra Y_i = X_i / A_i, fracción molar.
# Para n y p, A=1, así que Y = X.
Y0 = np.zeros(bbn.nnuc)

def find_index(A, Z, required=True):
    matches = [i for i, (a, z) in enumerate(zip(bbn.A, bbn.Z)) if int(a) == int(A) and int(z) == int(Z)]
    if not matches:
        if required:
            raise RuntimeError(f"No se encontró núcleo con A={A}, Z={Z}. Revisa la red.")
        return None
    return matches[0]

idx_n   = find_index(1, 0)
idx_p   = find_index(1, 1)
idx_d   = find_index(2, 1, required=False)
idx_t   = find_index(3, 1, required=False)
idx_he3 = find_index(3, 2, required=False)
idx_he4 = find_index(4, 2, required=False)
idx_li7 = find_index(7, 3, required=False)
idx_be7 = find_index(7, 4, required=False)

n_over_p = 1.0 / 7.0
Y0[idx_p] = 1.0 / (1.0 + n_over_p)
Y0[idx_n] = n_over_p / (1.0 + n_over_p)

print("Índice n:", idx_n, bbn.names[idx_n])
print("Índice p:", idx_p, bbn.names[idx_p])
print("Y_n/Y_p:", Y0[idx_n]/Y0[idx_p])
print("Suma bariónica inicial sum(A*Y):", np.sum(bbn.A * Y0))

In [ ]:
# RHS con T(t), rho(t) variables.
# No usamos screening para este primer ejercicio mínimo.

def rhs_var(t, Y):
    return bbn.rhs(t, Y, rho_of_t(t), T_of_t(t))

def jac_var(t, Y):
    return bbn.jacobian(t, Y, rho_of_t(t), T_of_t(t))

t_start = float(t_nodes[0])
t_end = float(t_nodes[-1])

# Mallado de salida: incluye exactamente los snapshots.
t_eval = np.unique(np.concatenate([
    np.geomspace(t_start, t_end, 400),
    t_nodes
]))

sol = solve_ivp(
    rhs_var,
    (t_start, t_end),
    Y0,
    method="BDF",
    jac=jac_var,
    t_eval=t_eval,
    dense_output=True,
    rtol=1.0e-8,
    atol=1.0e-20,
)

print("success:", sol.success)
print("message:", sol.message)
print("n_steps:", len(sol.t))

In [ ]:
# Pasamos de Y a X = A*Y y guardamos toda la evolución.
Y_hist = sol.y.T
X_hist = Y_hist * bbn.A[np.newaxis, :]

evolution = pd.DataFrame({
    "t_s": sol.t,
    "T9": T9_of_t(sol.t),
    "T_K": T_of_t(sol.t),
    "rho_g_cm3": rho_of_t(sol.t),
    "baryon_sum": np.sum(X_hist, axis=1),
})

for i, name in enumerate(bbn.names):
    clean = name.replace(" ", "")
    evolution[f"Y_{clean}"] = Y_hist[:, i]
    evolution[f"X_{clean}"] = X_hist[:, i]

evolution.to_csv("bbn_evolution.csv", index=False)
evolution.head()

In [ ]:
# Tabla en los tres snapshots exactos.
def eval_solution_at(t):
    Y = sol.sol(t)
    # Evita redondeos negativos diminutos en trazas.
    Y = np.where(np.abs(Y) < 1e-99, 0.0, Y)
    X = Y * bbn.A
    return Y, X

rows = []

for t in t_nodes:
    Y, X = eval_solution_at(t)
    YH = Y[idx_p]
    row = {
        "t_s": t,
        "T9": float(T9_of_t(t)),
        "rho_g_cm3": float(rho_of_t(t)),
        "baryon_sum": float(np.sum(X)),
        "Y_n/Y_p": float(Y[idx_n] / Y[idx_p]) if Y[idx_p] > 0 else np.nan,
    }

    # Magnitudes de interés cosmológico.
    row["D/H"] = float(Y[idx_d] / YH) if idx_d is not None and YH > 0 else np.nan
    row["T/H"] = float(Y[idx_t] / YH) if idx_t is not None and YH > 0 else np.nan
    row["He3/H"] = float(Y[idx_he3] / YH) if idx_he3 is not None and YH > 0 else np.nan
    row["X_He4"] = float(X[idx_he4]) if idx_he4 is not None else np.nan
    row["Li7/H"] = float(Y[idx_li7] / YH) if idx_li7 is not None and YH > 0 else np.nan
    row["Be7/H"] = float(Y[idx_be7] / YH) if idx_be7 is not None and YH > 0 else np.nan
    if idx_li7 is not None and idx_be7 is not None and YH > 0:
        row["Li7_plus_Be7_over_H"] = float((Y[idx_li7] + Y[idx_be7]) / YH)
    else:
        row["Li7_plus_Be7_over_H"] = np.nan

    # También guardamos X de todos los núcleos.
    for i, name in enumerate(bbn.names):
        clean = name.replace(" ", "")
        row[f"X_{clean}"] = float(X[i])

    rows.append(row)

snapshot_results = pd.DataFrame(rows)
snapshot_results.to_csv("bbn_snapshot_results.csv", index=False)
snapshot_results

In [ ]:
# Energía liberada total aproximada entre t=50 s y t=1000 s.
# Esta función está en el módulo generado por pynucastro.
dY_total = sol.y[:, -1] - Y0
E_erg_g = bbn.energy_release(dY_total)
print(f"Energy release = {E_erg_g:.6e} erg/g")

Archivos generados:

- `bbn_evolution.csv`: evolución temporal.
- `bbn_snapshot_results.csv`: tabla mínima para discutir el ejercicio.